# Notebook 4 — Kibana Exploration

*Hands-on time: ~25 minutes*

In this final notebook, we step into **Kibana** — the visual UI for ElasticSearch.
Most of this notebook is a guided walkthrough with screenshots and instructions;
the code cells prepare the data and verify your Kibana setup.

You will:
1. Access Kibana and navigate the interface
2. Use Dev Tools Console to run queries
3. Create a Data View for our index
4. Explore data in Discover
5. Build visualizations with Lens
6. Assemble a Dashboard

**Prerequisites:** Notebooks 1–3 completed, Docker services running.

---
## 0. Verify Kibana is Running

In [ ]:
import urllib.request
import json

KIBANA_URL = "http://localhost:5601"

try:
    req = urllib.request.Request(f"{KIBANA_URL}/api/status")
    with urllib.request.urlopen(req, timeout=10) as resp:
        status = json.loads(resp.read())
    print(f"Kibana status: {status['status']['overall']['level']}")
    print(f"Version:       {status['version']['number']}")
    print(f"\n✅ Kibana is running at {KIBANA_URL}")
except Exception as e:
    print(f"❌ Cannot reach Kibana at {KIBANA_URL}: {e}")
    print("Make sure Docker services are running: docker compose up -d")

In [ ]:
# Verify our index has data
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200")
INDEX_NAME = "neuroimaging"

count = client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' has {count['count']} documents")
assert count['count'] > 0, "No data! Run Notebook 1 first."

---
## 1. Navigate to Kibana

Open your browser and go to: **http://localhost:5601**

### First-time setup
1. Kibana may show a Welcome page — click **"Explore on my own"**
2. If prompted about sample data, skip it — we have our own data

---
## 2. Dev Tools Console

The Dev Tools Console is the fastest way to interact with ES from Kibana.

### How to access:
1. Click the **hamburger menu** (☰) → **Management** → **Dev Tools**
2. Or navigate directly to: **http://localhost:5601/app/dev_tools#/console**

### Try these queries in the Console:

```json
// Check cluster health
GET _cluster/health

// Count documents in our index
GET neuroimaging/_count

// View the index mapping
GET neuroimaging/_mapping

// Simple search
GET neuroimaging/_search
{
  "query": {
    "term": { "suffix": "bold" }
  },
  "size": 3
}

// Aggregation
GET neuroimaging/_search
{
  "size": 0,
  "aggs": {
    "by_suffix": {
      "terms": { "field": "suffix" }
    }
  }
}
```

> **Tip:** Place your cursor on a query and press **Ctrl+Enter** (or click the ▶ button) to execute it.

---
## 3. Create a Data View

A **Data View** (formerly Index Pattern) tells Kibana which ES index to visualize.

### Steps:
1. ☰ → **Stack Management** → **Data Views** (under Kibana section)
2. Click **"Create data view"**
3. Settings:
   - **Name**: `neuroimaging`
   - **Index pattern**: `neuroimaging`
   - **Timestamp field**: — (select \"I don't want to use a time filter\" since our data has no timestamps)
4. Click **Save data view to Kibana**

You should see all your fields listed with their types (keyword, float, dense_vector, etc.).

---
## 4. Discover — Explore Raw Data

### Steps:
1. ☰ → **Analytics** → **Discover**
2. Select the **neuroimaging** data view from the dropdown (top-left)
3. You should see all documents listed

### Try these explorations:

**Add columns:** Click the **+** button next to field names in the left sidebar:
- `subject`, `suffix`, `task`, `RepetitionTime`, `Manufacturer`

**Filter:** Click **+ Add filter** at the top:
- Field: `suffix`, Operator: `is`, Value: `bold` → Apply
- This shows only BOLD scans

**Search bar:** Type in the KQL search bar at the top:
- `suffix: bold AND RepetitionTime > 1.5`
- `subject: "01" OR subject: "02"`

**Inspect a document:** Click the expand arrow (▶) on any row to see all fields.

---
## 5. Build Visualizations with Lens

Lens is Kibana's drag-and-drop visualization builder.

### Visualization 1: Pie Chart — Scans by Suffix

1. ☰ → **Analytics** → **Visualize Library** → **Create visualization** → **Lens**
2. Select the **neuroimaging** data view
3. Change chart type to **Pie**
4. Drag `suffix` to the **Slice by** area
5. The metric should auto-set to Count
6. Click **Save** → name it **"Scans by Type"**

### Visualization 2: Bar Chart — Scans by Subject

1. Create new Lens visualization
2. Chart type: **Bar vertical**
3. Horizontal axis: drag `subject`
4. Vertical axis: Count (default)
5. Break down by: drag `suffix` to the **Breakdown** area
6. Save as **"Scans per Subject"**

### Visualization 3: Histogram — RepetitionTime Distribution

1. Create new Lens visualization
2. Chart type: **Bar vertical**
3. Horizontal axis: drag `RepetitionTime` → configure as a histogram
4. Vertical axis: Count
5. Save as **"TR Distribution"**

### Visualization 4: Data Table — Subject Overview

1. Create new Lens visualization
2. Chart type: **Table**
3. Rows: `subject`
4. Metrics: Count, then add Unique Count of `suffix`
5. Save as **"Subject Summary Table"**

---
## 6. Create a Dashboard

1. ☰ → **Analytics** → **Dashboard** → **Create dashboard**
2. Click **Add from library**
3. Add all 4 visualizations you just created:
   - Scans by Type (pie)
   - Scans per Subject (bar)
   - TR Distribution (histogram)
   - Subject Summary Table
4. **Resize and arrange** the panels by dragging
5. **Add a filter** at the top to make it interactive:
   - Click on a pie slice → the whole dashboard filters!
6. **Save** the dashboard as **"Neuroimaging Overview"**

> **Key insight:** Clicking any element in any visualization filters ALL other
> panels on the dashboard. This linked filtering is one of Kibana's most
> powerful features for exploratory data analysis.

---
## 7. Bonus: kNN Queries in Dev Tools

You can run vector search directly from the Kibana Dev Tools Console.
The cell below generates a query vector you can copy-paste into Kibana.

In [ ]:
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer("all-MiniLM-L6-v2")

query_text = "brain activation during risk-taking"
query_vec = model.encode(query_text).tolist()

# Build the kNN query for copy-paste into Kibana Dev Tools
kibana_query = {
    "knn": {
        "field": "metadata_embedding",
        "query_vector": query_vec,
        "k": 5,
        "num_candidates": 50
    },
    "_source": {
        "excludes": ["metadata_embedding"]
    }
}

print("Copy this into Kibana Dev Tools Console:")
print("="*50)
print(f"GET neuroimaging/_search")
print(json.dumps(kibana_query, indent=2))

### Try it:
1. Go to Dev Tools Console
2. Paste the query above
3. Execute with **Ctrl+Enter**
4. Inspect the results — you'll see the same semantic search results
   as in Notebook 3, but through the Kibana interface

> This is the workflow for production: generate embedding vectors in your
> application code, then send kNN queries to ES.

---
## Verification Checklist

Before finishing, verify you've completed these steps:

In [ ]:
checklist = [
    "Accessed Kibana at http://localhost:5601",
    "Ran a query in Dev Tools Console",
    "Created a Data View for 'neuroimaging'",
    "Explored documents in Discover with filters",
    "Built a Pie chart (Scans by Type)",
    "Built a Bar chart (Scans per Subject)",
    "Built a Histogram (TR Distribution)",
    "Built a Data Table (Subject Summary)",
    "Created a Dashboard combining all visualizations",
    "Tested linked filtering on the Dashboard",
    "Ran a kNN query in Dev Tools Console",
]

print("🏁 Course Completion Checklist")
print("="*40)
for i, item in enumerate(checklist, 1):
    print(f"  [ ] {i:2d}. {item}")
print(f"\nTotal items: {len(checklist)}")

---
## Summary — Course Complete!

Congratulations! Across all 4 notebooks you have:

| Notebook | What you learned |
| --- | --- |
| **NB1** | BIDS exploration with pybids, ES index mapping, embedding generation, bulk ingestion |
| **NB2** | BM25 full-text search, term/range filters, bool compound queries, aggregations |
| **NB3** | kNN vector search, filtered kNN, hybrid search, parameter tuning |
| **NB4** | Kibana navigation, Dev Tools, Discover, Lens visualizations, Dashboards |

### What you've built:
- A **vector-enabled search engine** for neuroimaging metadata
- That supports **keyword**, **semantic**, and **hybrid** search
- With a **Kibana dashboard** for visual exploration
- All running locally in **Docker**

See `docs/05-next-steps.md` for ideas on scaling this to production.